In [ ]:
import requests
import json
import pandas as pd
import time
import folium

from datetime import datetime

In [ ]:
# PEYTON RUN
filepath = "/Users/peyton/Desktop/TicketmasterAPIKey.txt"

In [ ]:
# KATRIEL RUN
filepath = "/Users/katriellayam/Downloads/ticketmaster_project.txt"

In [ ]:
# LEAH RUN
filepath = "/Users/leahaviles/Downloads/ticket_masterAPIKey.txt"

In [ ]:
# JOY RUN
filepath = "/Users/joy/Downloads/ticketmaster api key.txt"

In [ ]:
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key(filepath)

type(key)

In [ ]:
url = f"https://app.ticketmaster.com/discovery/v2/venues.json?apikey={key}"

response = requests.get(url)

response.raise_for_status()

test_data = response.json()

test_data

In [ ]:
venues = test_data['_embedded']['venues']

df = pd.DataFrame([{
    'id': v.get('id'),
    'name': v.get('name'),
    'address': v.get('address', {}).get('line1'),
    'city': v.get('city', {}).get('name'),
    'state': v.get('state', {}).get('stateCode'),
    'latitude': v.get('location', {}).get('latitude'),
    'longitude': v.get('location', {}).get('longitude'),
    'postal_code': v.get('postalCode'),
    'upcoming_events_total': v.get('upcomingEvents', {}).get('_total'),
} for v in venues])

df = df.sort_values(by='upcoming_events_total', ascending=False) # order df in terms of most upcoming events
df = df.drop(columns='id') # remove id column

df

In [ ]:
m = folium.Map(location=[38, -95], zoom_start=4)

for _, row in df.iterrows():
    if row["latitude"] is None or row["longitude"] is None:
        continue

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['address']}<br>"
        f"{row['city']}, {row['state']} {row['postal_code']}<br>"
        f"Upcoming Events: {row['upcoming_events_total']}"
    )

    folium.Marker(
        location=[float(row["latitude"]), float(row["longitude"])],
        popup=folium.Popup(popup_text, max_width=200),
        icon=folium.Icon(color="darkblue", icon="ticket", prefix="fa")
    ).add_to(m)

m.save("venues.html")
m

In [ ]:
# fetch events per month
import time

all_events = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while True:
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()

        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})
            all_events.append({
                "name": e.get("name"),
                "date": e.get("dates", {}).get("start", {}).get("localDate"),
                "venue": venue.get("name"),
                "city": venue.get("city", {}).get("name"),
                "state": venue.get("state", {}).get("stateCode"),
                "latitude": loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "url": e.get("url"),
                "month": start[:7],
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events)} events so far")

        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(0.25)  # rate limit buffer

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"] = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

print(f"\nTotal events: {len(events_df)}")

In [ ]:
# this time we fetch 'classifications' gives if its music, sport, etc.
all_events = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while True:
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()

        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})

            # extract classifications
            classifications = e.get("classifications", [{}])
            primary = classifications[0] if classifications else {}
            segment  = primary.get("segment", {}).get("name")
            genre    = primary.get("genre", {}).get("name")
            subgenre = primary.get("subGenre", {}).get("name")

            all_events.append({
                "name":     e.get("name"),
                "date":     e.get("dates", {}).get("start", {}).get("localDate"),
                "venue":    venue.get("name"),
                "city":     venue.get("city", {}).get("name"),
                "state":    venue.get("state", {}).get("stateCode"),
                "latitude":  loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "url":      e.get("url"),
                "month":    start[:7],
                "segment":  segment,
                "genre":    genre,
                "subgenre": subgenre,
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events)} events so far")

        if page + 1 >= total_pages:
            break

        page += 1
        time.sleep(0.25)

events_df = pd.DataFrame(all_events)
events_df = events_df.dropna(subset=["latitude", "longitude"])
events_df["latitude"]  = events_df["latitude"].astype(float)
events_df["longitude"] = events_df["longitude"].astype(float)

print(f"\nTotal events: {len(events_df)}")
print(events_df[["segment", "genre", "subgenre"]].value_counts().head(20))

In [ ]:
import sqlite3

# create in-memory db and load df
conn = sqlite3.connect(":memory:")
events_df.to_sql("events", conn, index=False, if_exists="replace")
df.to_sql("venues", conn, index=False, if_exists="replace")

# events by segment
pd.read_sql("""
    SELECT segment, COUNT(*) as count
    FROM events
    GROUP BY segment
    ORDER BY count DESC
""", conn)

In [ ]:
# only include events in mainland US
events_df = pd.read_sql("""
    SELECT *
    FROM events
    WHERE latitude BETWEEN 24.5 AND 49.5
    AND longitude BETWEEN -125.0 AND -66.5
""", conn)

print(f"Events before: {len(events_df)}")

# update the sql table
events_df.to_sql("events", conn, index=False, if_exists="replace")

print(f"Events after: {len(events_df)}")

events_df

In [ ]:
# only include events in US
# categorize events by types of color

m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

# color by segment
segment_colors = {
    "Music": "blue",
    "Sports": "red",
    "Arts & Theatre": "green",
    "Miscellaneous": "gray",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in events_df.iterrows():
    month = row["month"]
    if month not in groups:
        continue

    color = segment_colors.get(row["segment"], "gray")

    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}<br>"
        f"Segment: {row['segment']}<br>"
        f"Genre: {row['genre']}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [ ]:
# make chorolopleth map colored by most popular genre in each state
import plotly.express as px

# find most popular genre per state, excluding vague ones
top_genre_by_state = (
    events_df[~events_df["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["state", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .drop_duplicates(subset="state")  # keep top genre per state
)

fig = px.choropleth(
    top_genre_by_state,
    locations="state",
    locationmode="USA-states",
    color="genre",
    scope="usa",
    hover_name="state",
    hover_data={"count": True, "genre": True},
    title="Most Popular Event Genre by State (2026)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_layout(
    legend_title="Top Genre",
    title_font_size=16,
)

fig.show()

In [ ]:
# can toggle event locations per month
m = folium.Map(location=[38, -95], zoom_start=4)

month_labels = {
    "2026-03": "March", "2026-04": "April", "2026-05": "May",
    "2026-06": "June", "2026-07": "July", "2026-08": "August",
    "2026-09": "September", "2026-10": "October", "2026-11": "November",
    "2026-12": "December",
}

groups = {}
for month_key, label in month_labels.items():
    groups[month_key] = folium.FeatureGroup(name=label, show=False)

for _, row in events_df.iterrows():
    month = row["month"]
    if month not in groups:
        continue
    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"{row['venue']}<br>"
        f"{row['city']}, {row['state']}<br>"
        f"Date: {row['date']}"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="darkblue",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(groups[month])

for group in groups.values():
    group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("events_by_month.html")
m

In [ ]:
# plot categories
import matplotlib.pyplot as plt

genre_counts = (
    events_df[~events_df["genre"].isin(["Undefined", "Miscellaneous", "Other"])]
    .groupby(["segment", "genre"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

# color by segment
segment_colors = {
    "Music": "steelblue",
    "Sports": "firebrick",
    "Arts & Theatre": "mediumseagreen",
    "Miscellaneous": "gray",
}
colors = genre_counts["segment"].map(segment_colors).fillna("gray")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    genre_counts["genre"] + " (" + genre_counts["segment"] + ")",
    genre_counts["count"],
    color=colors
)

ax.set_xlabel("Number of Events")
ax.set_title("Top 20 Event Genres on Ticketmaster (2026)", fontsize=14)
ax.invert_yaxis()

# legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in segment_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# data cleaning!! so awesome
# some events have abbreviations, some have full names like "Texas" and "TX" are both options
# we coerce them to be the same
state_name_to_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC",
}

def normalize_state(val):
    if val is None:
        return None
    val = val.strip()
    if len(val) == 2:
        return val.upper()
    return state_name_to_abbr.get(val, None)

events_df["state"] = events_df["state"].apply(normalize_state)
events_df.to_sql("events", conn, index=False, if_exists="replace")

In [ ]:
events_per_state = (
    pd.read_sql("""
        SELECT state, COUNT(*) as event_count
        FROM events
        WHERE state IS NOT NULL
        GROUP BY state
        ORDER BY event_count DESC
    """, conn)
)

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(events_per_state["state"], events_per_state["event_count"], color="steelblue")
ax.set_xlabel("State")
ax.set_ylabel("Number of Events")
ax.set_title("Number of Events per State (2026)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Below I am going to analyze the proximity of events to large US cities.

In [ ]:
# intiialize coords of large US cities
major_cities = {
    "New York":      (40.7128, -74.0060),
    "Los Angeles":   (34.0522, -118.2437),
    "Chicago":       (41.8781, -87.6298),
    "Houston":       (29.7604, -95.3698),
    "Phoenix":       (33.4484, -112.0740),
    "Philadelphia":  (39.9526, -75.1652),
    "San Antonio":   (29.4241, -98.4936),
    "San Diego":     (32.7157, -117.1611),
    "Dallas":        (32.7767, -96.7970),
    "San Jose":      (37.3382, -121.8863),
    "Austin":        (30.2672, -97.7431),
    "Jacksonville":  (30.3322, -81.6557),
    "Seattle":       (47.6062, -122.3321),
    "Denver":        (39.7392, -104.9903),
    "Nashville":     (36.1627, -86.7816),
    "Boston":        (42.3601, -71.0589),
    "Atlanta":       (33.7490, -84.3880),
    "Miami":         (25.7617, -80.1918),
    "Minneapolis":   (44.9778, -93.2650),
    "Portland":      (45.5051, -122.6750),
}

cities_df = pd.DataFrame(
    [(city, lat, lon) for city, (lat, lon) in major_cities.items()],
    columns=["city_name", "city_lat", "city_lon"]
)

In [ ]:
# calculate the distance to the nearest US city
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def nearest_city(lat, lon):
    best_city, best_dist = None, float("inf")
    for _, row in cities_df.iterrows():
        d = haversine(lat, lon, row["city_lat"], row["city_lon"])
        if d < best_dist:
            best_dist = d
            best_city = row["city_name"]
    return best_city, round(best_dist, 1)

# apply to events_df
events_df[["nearest_city", "dist_to_city_miles"]] = events_df.apply(
    lambda row: nearest_city(row["latitude"], row["longitude"]),
    axis=1, result_type="expand"
)

events_df.to_sql("events", conn, index=False, if_exists="replace")

In [ ]:
# distribution of distances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of distances
axes[0].hist(events_df["dist_to_city_miles"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Distance to Nearest Major City (miles)")
axes[0].set_ylabel("Number of Events")
axes[0].set_title("How Far Are Events from Major Cities?")

# % of events within thresholds
thresholds = [10, 25, 50, 100, 200]
pcts = [
    (events_df["dist_to_city_miles"] <= t).mean() * 100
    for t in thresholds
]
axes[1].bar([f"≤{t} mi" for t in thresholds], pcts, color="mediumseagreen")
axes[1].set_ylabel("% of Events")
axes[1].set_title("Cumulative % of Events Within Distance of a Major City")
axes[1].set_ylim(0, 100)
for i, v in enumerate(pcts):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# events per nearest city
pd.read_sql("""
    SELECT nearest_city, COUNT(*) as event_count,
           ROUND(AVG(dist_to_city_miles), 1) as avg_dist
    FROM events
    GROUP BY nearest_city
    ORDER BY event_count DESC
""", conn)

Ok the analysis for the proximity to major cities is between these boxes.

now we will look at prices

In [ ]:
# fetch prices
all_events_price = []

months = [
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", "2026-06-30"),
    ("2026-07-01", "2026-07-31"),
    ("2026-08-01", "2026-08-31"),
    ("2026-09-01", "2026-09-30"),
    ("2026-10-01", "2026-10-31"),
    ("2026-11-01", "2026-11-30"),
    ("2026-12-01", "2026-12-31"),
]

for start, end in months:
    page = 0
    while page < 5:  # cap at 5 pages per month to stay within rate limits
        url = (
            f"https://app.ticketmaster.com/discovery/v2/events.json"
            f"?apikey={key}"
            f"&startDateTime={start}T00:00:00Z"
            f"&endDateTime={end}T23:59:59Z"
            f"&countryCode=US"
            f"&size=200"
            f"&page={page}"
        )
        response = requests.get(url)
        data = response.json()
        events = data.get("_embedded", {}).get("events", [])
        if not events:
            break

        for e in events:
            venue = e.get("_embedded", {}).get("venues", [{}])[0]
            loc = venue.get("location", {})

            # classifications
            classifications = e.get("classifications", [{}])
            primary = classifications[0] if classifications else {}
            segment  = primary.get("segment", {}).get("name")
            genre    = primary.get("genre", {}).get("name")

            # price ranges
            price_ranges = e.get("priceRanges", [])
            price_min = price_ranges[0].get("min") if price_ranges else None
            price_max = price_ranges[0].get("max") if price_ranges else None

            all_events_price.append({
                "name":      e.get("name"),
                "date":      e.get("dates", {}).get("start", {}).get("localDate"),
                "venue":     venue.get("name"),
                "city":      venue.get("city", {}).get("name"),
                "state":     venue.get("state", {}).get("stateCode"),
                "latitude":  loc.get("latitude"),
                "longitude": loc.get("longitude"),
                "month":     start[:7],
                "segment":   segment,
                "genre":     genre,
                "price_min": price_min,
                "price_max": price_max,
            })

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"{start[:7]} — page {page+1}/{total_pages} — {len(all_events_price)} events so far")
        if page + 1 >= total_pages:
            break
        page += 1
        time.sleep(0.25)

price_df = pd.DataFrame(all_events_price)

# clean up
price_df["state"] = price_df["state"].apply(normalize_state)
price_df = price_df.dropna(subset=["latitude", "longitude", "price_min", "price_max"])
price_df["latitude"]  = price_df["latitude"].astype(float)
price_df["longitude"] = price_df["longitude"].astype(float)

# filter to continental US
price_df = price_df[
    price_df["latitude"].between(24.5, 49.5) &
    price_df["longitude"].between(-125.0, -66.5)
]

# load into sqlite
price_df.to_sql("events_price", conn, index=False, if_exists="replace")

print(f"\nEvents with price data: {len(price_df)}")
print(f"Events WITHOUT price data: {len(pd.DataFrame(all_events_price)) - len(price_df)}")

In [ ]:
# avg price by genre
pd.read_sql("""
    SELECT genre,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    WHERE genre NOT IN ('Undefined', 'Miscellaneous', 'Other')
    GROUP BY genre
    ORDER BY avg_min DESC
    LIMIT 15
""", conn)

In [ ]:
# avg price by state
pd.read_sql("""
    SELECT state,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY state
    ORDER BY avg_min DESC
""", conn)

In [ ]:
# avg price by segment
pd.read_sql("""
    SELECT segment,
           ROUND(AVG(price_min), 2) as avg_min,
           ROUND(AVG(price_max), 2) as avg_max,
           COUNT(*) as event_count
    FROM events_price
    GROUP BY segment
    ORDER BY avg_min DESC
""", conn)

there isn't much price information, i don't know if we should use it

In [ ]:
from folium.plugins import HeatMap
import numpy as np

from folium.plugins import HeatMap
import numpy as np

# ── 1. Density heatmap ───────────────────────────────────────────────────────
m = folium.Map(location=[38, -95], zoom_start=4)

heat_data = events_df[["latitude", "longitude"]].dropna().values.tolist()

HeatMap(
    heat_data,
    radius=12,
    blur=18,
    min_opacity=0.3,
    gradient={0.2: "blue", 0.5: "lime", 0.8: "orange", 1.0: "red"},
).add_to(m)

colormap = folium.LinearColormap(
    colors=["blue", "lime", "orange", "red"],
    index=[0.2, 0.5, 0.8, 1.0],
    vmin=0.2,
    vmax=1.0,
    caption="Event Density (low → high)"
)
colormap.add_to(m)

m.get_root().html.add_child(folium.Element("""
    <style>
        .colormap { position: fixed !important; top: 20px !important; right: 20px !important; bottom: auto !important; left: auto !important; }
    </style>
"""))

m.save("event_density_heatmap.html")
m

# ── 2. Grid + desert cells ───────────────────────────────────────────────────
GRID_DEG = 0.75

lat_bins = np.arange(24.5, 49.5, GRID_DEG)
lon_bins = np.arange(-125.0, -66.5, GRID_DEG)

events_df["lat_bin"] = pd.cut(events_df["latitude"], bins=lat_bins, labels=lat_bins[:-1]).astype(float)
events_df["lon_bin"] = pd.cut(events_df["longitude"], bins=lon_bins, labels=lon_bins[:-1]).astype(float)

grid = (
    events_df
    .groupby(["lat_bin", "lon_bin"], observed=True)
    .size()
    .reset_index(name="event_count")
)

LOW_THRESHOLD = grid["event_count"].quantile(0.20)
desert_grid = grid[grid["event_count"] <= LOW_THRESHOLD]

print(f"Desert threshold : ≤ {LOW_THRESHOLD:.0f} events per ~50-mile cell")
print(f"Desert cells     : {len(desert_grid)} of {len(grid)} total populated cells")

# ── 3. Desert map ────────────────────────────────────────────────────────────
m2 = folium.Map(location=[38, -95], zoom_start=4)

HeatMap(
    heat_data,
    radius=12,
    blur=18,
    min_opacity=0.2,
    gradient={0.2: "blue", 0.5: "lime", 0.8: "orange", 1.0: "red"},
    name="Event density",
    show=True,
).add_to(m2)

desert_group = folium.FeatureGroup(name="Event deserts (low access)", show=True)

for _, row in desert_grid.iterrows():
    center_lat = row["lat_bin"] + GRID_DEG / 2
    center_lon = row["lon_bin"] + GRID_DEG / 2
    folium.CircleMarker(
        location=[center_lat, center_lon],
        radius=6,
        color="crimson",
        fill=True,
        fill_color="crimson",
        fill_opacity=0.5,
        popup=folium.Popup(
            f"Only {row['event_count']:.0f} event(s) in this area", max_width=200
        ),
    ).add_to(desert_group)

desert_group.add_to(m2)

folium.LayerControl(collapsed=False).add_to(m2)

colormap2 = folium.LinearColormap(
    colors=["blue", "lime", "orange", "red"],
    index=[0.2, 0.5, 0.8, 1.0],
    vmin=0.2,
    vmax=1.0,
    caption="Event Density (low → high)"
)
colormap2.add_to(m2)

m2.get_root().html.add_child(folium.Element("""
    <style>
        .colormap { position: fixed !important; top: 20px !important; right: 20px !important; bottom: auto !important; left: auto !important; }
        .leaflet-top.leaflet-right { top: auto !important; bottom: 20px !important; }
    </style>
"""))

m2.save("event_desert_map.html")
m2